# 🎬 Movie Recap AI - Kaggle GPU Edition (100% Free)
Kaggle ပေါ်မှာ **Nvidia T4 GPU (16GB)** နဲ့ အခမဲ့ Run နိုင်တဲ့ အဆင်သင့် Notebook ဖြစ်ပါတယ်။

### ⚠️ အသုံးမပြုမီ အရေးကြီးသော အချက်များ (First-Time Setup):
1. ညာဘက်ခြမ်း **Notebook Options** ထဲက **Accelerator** မှာ **`GPU T4 x 2`** (သို့မဟုတ် `GPU P100`) ကို ရွေးပေးပါ။
2. **Internet** ကို **`Internet on`** (Toggle ON) သေချာ ပြောင်းပေးပါ။ *(အရေးကြီးသည်)*

*(မှတ်ချက် - ဤ Notebook ကို Run ပြီးပါက ကွန်ပျူတာ ပိတ်ထားလည်း အလုပ်လုပ်နိုင်ပါသည်)*

### 🔹 အဆင့် ၁: Project ရယူခြင်းနှင့် လိုအပ်သည်များ သွင်းယူခြင်း

In [ ]:
# Step 1: Clone Repository & Install Dependencies
import os

project_path = "/kaggle/working/ai-translate-agent"
if not os.path.exists(project_path):
    print("[*] First-time setup: Cloning AI-Movie-Translate...")
    !git clone https://github.com/paipai1999/ai-translate-agent.git {project_path}
else:
    print("[*] Updating to latest code...")
    %cd {project_path}
    !git reset --hard HEAD
    !git pull origin main

%cd {project_path}

# Create NVMe temp directory
os.makedirs("/kaggle/temp", exist_ok=True)

print("[*] Installing packages & system tools (FFmpeg, Myanmar Fonts)...")
!pip install -q -r requirements.txt
!apt-get update -qq
!apt-get install -y -qq ffmpeg fonts-noto-core fonts-noto-cjk fonts-sil-padauk
print("\n✅ Setup ပြီးမြောက်ပါပြီ! Step 2 သို့ ဆက်သွားပါ။")

### 🔹 အဆင့် ၂: Gemini API Key နှင့် Settings သတ်မှတ်ခြင်း

In [ ]:
import json
import os

# -----------------------------------------------------
# သင့်ရဲ့ Google AI Studio Gemini API Key ထည့်ပါ
# -----------------------------------------------------
GEMINI_API_KEYS = [
    "YOUR_GEMINI_API_KEY_HERE"
]

config_data = {
    "gemini": {
        "enabled": True,
        "api_keys": GEMINI_API_KEYS,
        "model": "gemini-3.5-flash-lite",
        "daily_limit_per_key": 500,
        "models": {
            "heavy": "gemini-3.7-flash",
            "workhorse": "gemini-3.5-flash-lite",
            "polish": "gemini-3.5-flash-lite"
        }
    },
    "pipeline": {
        "language": "burmese",
        "whisper_model": "small",
        "parallel_processing": True,
        "use_demucs": True
    },
    "paths": {
        "temp_dir": "/kaggle/temp",
        "output_dir": "outputs"
    },
    "voice": {
        "enabled": True,
        "voice_mode": "dynamic",
        "tts_voice_mm": "my-MM-ThihaNeural",
        "tts_voice_en": "en-US-GuyNeural",
        "tts_voice": "my-MM-ThihaNeural",
        "tts_rate_mm": "+8%",
        "tts_rate_en": "+15%"
    },
    "subtitle_overlay": {
        "enabled": True,
        "font_name": "Padauk",
        "font_size": 40,
        "bold": True,
        "border_style": 3,
        "outline_width": 3,
        "margin_bottom": 50,
        "max_chars_per_line": 28
    }
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=4)

print("✅ config.json created successfully!")
print("🚀 Ready to Run!")

### 🔹 အဆင့် ၃ (Option A): Command Line ဖြင့် တိုက်ရိုက် Run ခြင်း
*ရုပ်ရှင် URL ထည့်ပြီး တစ်ခါတည်း Recap လုပ်ချင်လျှင် ဤ Cell ကို Run ပါ။*

In [ ]:
# YouTube Video URL သို့မဟုတ် Direct Video URL ထည့်ပါ
VIDEO_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"

# Start Recap Pipeline!
!python main.py -i "$VIDEO_URL" --blocks 5

### 🔹 အဆင့် ၄ (Option B): Web UI Dashboard ဖွင့်ပြီး Browser/ဖုန်းမှ အသုံးပြုခြင်း
*Web UI သုံးချင်ပါက ဤ Cell ကို Run ပါ။ Cloudflare Link (.trycloudflare.com) ထွက်လာပါက နှိပ်ပြီး UI ဖွင့်သုံးနိုင်ပါသည်။*

In [ ]:
# Setup Cloudflare Tunnel for Web UI
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess
import time

print("🧹 Cleaning up old processes...")
!pkill -f "python web_ui.py" || true
!pkill -f "cloudflared" || true

print("🚀 Starting Web UI in background...")
flask_proc = subprocess.Popen(
    ['python', 'web_ui.py'],
    stdout=open('/kaggle/working/web_ui.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(5)

print("🌐 Creating secure Cloudflare public URL...")
print("👇 အောက်ပါ Output ထဲတွင် ပေါ်လာမည့် '.trycloudflare.com' Link ကို နှိပ်ပြီး ဝင်သုံးပါ 👇\n")
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:5000

### 🔹 အဆင့် ၅: ပြီးစီးသွားသော ဗီဒီယိုများကို Download လုပ်ခြင်း

In [ ]:
import glob
import os
from IPython.display import FileLink, display

output_files = glob.glob("/kaggle/working/ai-translate-agent/outputs/**/*.mp4", recursive=True)

if output_files:
    print("🎉 ပြီးစီးသွားသော ဗီဒီယိုများ:\n")
    for f in output_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"📁 {os.path.basename(f)} ({size_mb:.1f} MB)")
        display(FileLink(os.path.relpath(f, start="/kaggle/working")))
else:
    print("ℹ️ outputs folder ထဲတွင် ဗီဒီယို မရှိသေးပါ။")